# WideResNet-50-2: CE versus focal loss

Both experiments call the project's tested training scripts. The model is an ImageNet-pretrained WideResNet-50-2, with early stages frozen, layer3/layer4 trainable, and one 2048-to-3 classifier. There is no 64-D projection in these baselines.

Thesis Table 4.7 (PDF page 101 / printed page 79): CE accuracy **54.3 ± 2.4%**, Parkinson precision **0.67 ± 0.05**, F1 **0.57 ± 0.06**; focal accuracy **62.5 ± 2.1%**, precision **0.80 ± 0.06**, F1 **0.62 ± 0.05**. These are published results, not results generated by this notebook.

The thesis describes conventional noise/rotation/flipping and replacement sampling but omits exact strengths and training settings. Values below are documented project choices. PNG-to-time-colour equivalence and the final weighted ensemble remain unverified. This is a thesis-guided baseline, not a certified exact reproduction.

For Colab: first run `python prepare_colab_upload.py` locally and upload the resulting `wideresnet_development.zip` in the next cell. It includes all 25 development folds and no test images. Choose a GPU runtime. Local Jupyter also works from this repository.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

candidates = [Path.cwd(), Path.cwd().parent, Path('/content/ParkinsonAiResearch')]
PROJECT_DIR = next((p for p in candidates if (p / 'train.py').is_file()), None)
if PROJECT_DIR is None:
    from google.colab import files
    uploaded = files.upload()  # select wideresnet_development.zip
    bundles = [Path(name) for name in uploaded if name.endswith('.zip')]
    if len(bundles) != 1:
        raise ValueError('Upload exactly one wideresnet_development.zip archive.')
    PROJECT_DIR = Path('/content/ParkinsonAiResearch')
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundles[0]) as archive:
        for name in archive.namelist():
            if not (PROJECT_DIR / name).resolve().is_relative_to(PROJECT_DIR.resolve()):
                raise ValueError('Archive contains an unsafe path')
        archive.extractall(PROJECT_DIR)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
print('Project:', PROJECT_DIR)
print('Device:', 'cuda' if torch.cuda.is_available() else 'CPU/local auto')


Start with fold 1 to check that both experiments finish. Its few validation images cannot establish which loss is better. Then set `FOLD_END = 25` and rerun the configuration and training cells; `--resume` reuses completed folds. Full comparison requires all 25 fold reports, including the explicit empty-validation report for fold 22.

If using Colab, change `OUTPUT_BASE` to a persistent mounted folder before a long run, or save the checkpoints yourself before the runtime ends. Resume requires both the same dataset/code configuration and the saved `last.pt` files.


In [ ]:
FOLD_START = 1
FOLD_END = 1  # set to 25 after the pilot succeeds
EPOCHS = 15
BATCH_SIZE = 8
SEED = 42
HEAD_LR = 1e-3
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 1e-4
FOCAL_GAMMA = 2.0
FOCAL_ALPHA_MODE = 'uniform'
AUGMENTATION = 'standard'
SAMPLING = 'class_balanced'
ROTATION_DEGREES = 10
HFLIP_PROB = 0.5
NOISE_STD = 0.01
DEVICE = 'cuda' if torch.cuda.is_available() else 'auto'
OUTPUT_BASE = PROJECT_DIR / 'outputs' / 'baselines'
MANIFEST = PROJECT_DIR / 'datasets' / 'annotations' / 'fold_manifest.csv'

COMMON = [
    '--fold-start', str(FOLD_START), '--fold-end', str(FOLD_END),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--image-size', '224',
    '--pretrained', '--freeze-policy', 'early', '--label-source', 'consensus',
    '--lr', str(HEAD_LR), '--backbone-lr', str(BACKBONE_LR),
    '--weight-decay', str(WEIGHT_DECAY), '--seed', str(SEED),
    '--augmentation', AUGMENTATION, '--sampling', SAMPLING,
    '--rotation-degrees', str(ROTATION_DEGREES), '--hflip-prob', str(HFLIP_PROB),
    '--noise-std', str(NOISE_STD), '--num-workers', '0', '--device', DEVICE,
    '--manifest', str(MANIFEST), '--root', str(PROJECT_DIR), '--resume',
]

def run_loss(loss):
    command = [sys.executable, '-u', 'train_all_folds.py', *COMMON,
               '--loss', loss, '--checkpoint-dir', str(OUTPUT_BASE / loss / 'checkpoints')]
    if loss == 'focal':
        command += ['--focal-gamma', str(FOCAL_GAMMA), '--focal-alpha-mode', FOCAL_ALPHA_MODE]
    subprocess.run(command, check=True)


Run WideResNet + cross-entropy. The first pretrained run downloads the ImageNet weights if needed.

In [ ]:
run_loss('cross_entropy')


Run WideResNet + focal loss with the same architecture, fold schedule, seed, optimizer and budget.

In [ ]:
run_loss('focal')


Pool development predictions. The report contains accuracy, Parkinson precision/recall/F1, macro F1, per-class support and the confusion matrix. A pilot report is marked incomplete. Pooled development results are not the thesis's final weighted test-ensemble result.

In [ ]:
import json
for loss in ('cross_entropy', 'focal'):
    output = OUTPUT_BASE / loss
    command = [sys.executable, 'aggregate_validation.py', '--loss', loss,
               '--predictions-dir', str(output / 'validation_predictions'),
               '--output-dir', str(output / 'reports'), '--manifest', str(MANIFEST),
               '--seed', str(SEED), '--freeze-policy', 'early']
    if loss == 'focal':
        command += ['--focal-alpha-mode', FOCAL_ALPHA_MODE]
    if FOLD_START == 1 and FOLD_END == 25:
        command += ['--require-complete']
    subprocess.run(command, check=True)
    report = json.loads((output / 'reports' / 'oof_validation_summary.json').read_text())
    metrics = report['pooled_oof_metrics']
    print(loss, report['status'], 'OOF images:', report['predicted_oof_images'])
    if metrics:
        print('Accuracy:', metrics['accuracy'], 'Parkinson:', metrics['per_class']['parkinson'],
              'Macro F1:', metrics['macro_f1_all_classes'])


Save a small results bundle to share for the next hyperparameter review. It excludes large model checkpoints; save those separately to continue training.

In [ ]:
results_zip = OUTPUT_BASE / 'baseline_results.zip'
with zipfile.ZipFile(results_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for loss in ('cross_entropy', 'focal'):
        for file in (OUTPUT_BASE / loss).rglob('*'):
            if file.is_file() and file.suffix in {'.json', '.csv'}:
                archive.write(file, file.relative_to(OUTPUT_BASE))
print(results_zip)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(results_zip))


## Optional: repeat the complete development evaluation across seeds

The pilot above checks execution. For the evaluation used to compare configurations, the new workflow repeats all 25 prescribed holdouts for predetermined seeds, combines genuinely held-out predictions per seed, and reports image-weighted and subject-balanced metrics. Incomplete pairs contribute their available held-out subject; empty pairs contribute no predictions. Missing pair members are reported explicitly.

The following cell defaults to a **plan only**. Set `RUN_REPEATED_EVALUATION = True` after reviewing the plan and choosing persistent storage. Three seeds mean **75 model runs per loss, 150 for both losses**, independently of `FOLD_START`, `FOLD_END` and the single pilot `SEED`. The seed standard deviation measures training randomness, not a population confidence interval. Test data are never evaluated. See `README/DEVELOPMENT_EVALUATION.md`.


In [ ]:
REPEATED_SEEDS = [42, 43, 44]  # decide before looking at results
RUN_REPEATED_EVALUATION = False  # False: write plans and coverage only
REPEATED_OUTPUT = OUTPUT_BASE.parent / 'repeated_validation'

# Reuse the pilot's recipe, with the full fold schedule and seeds managed by the evaluator.
repeated_common = []
i = 0
while i < len(COMMON):
    if COMMON[i] in {'--seed', '--fold-start', '--fold-end'}:
        i += 2
    else:
        repeated_common.append(COMMON[i])
        i += 1

for loss in ('cross_entropy', 'focal'):
    command = [sys.executable, '-u', 'evaluate_development.py', *repeated_common,
               '--seeds', *map(str, REPEATED_SEEDS), '--loss', loss,
               '--output-dir', str(REPEATED_OUTPUT / loss)]
    if loss == 'focal':
        command += ['--focal-gamma', str(FOCAL_GAMMA), '--focal-alpha-mode', FOCAL_ALPHA_MODE]
    if not RUN_REPEATED_EVALUATION:
        command.append('--dry-run')
    subprocess.run(command, check=True)
    print(loss, 'plan:', REPEATED_OUTPUT / loss / 'evaluation_plan.json')
    if RUN_REPEATED_EVALUATION:
        print('Combined report:', REPEATED_OUTPUT / loss / 'combined' / 'Report.md')

if RUN_REPEATED_EVALUATION:
    repeated_results_zip = REPEATED_OUTPUT / 'repeated_validation_results.zip'
    with zipfile.ZipFile(repeated_results_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in REPEATED_OUTPUT.rglob('*'):
            if path.is_file() and path.suffix in {'.json', '.csv', '.md'}:
                archive.write(path, path.relative_to(REPEATED_OUTPUT))
    print('Results bundle (checkpoints excluded):', repeated_results_zip)
    if 'google.colab' in sys.modules:
        from google.colab import files
        files.download(str(repeated_results_zip))
